# 08 — Grouped ML evaluation and stability diagnostics

This notebook evaluates the prepared wobble-welding dataset using the same baseline, Ridge and Extra Trees configurations as the initial model comparison. It reports fold-wise metrics, pooled out-of-fold metrics, and diagnostic plots. It does not perform hyperparameter tuning or claim external validation.

**Input:** `wobbling_ml_dataset.csv` from the initial modelling preparation.

**Validation:** five-fold `GroupKFold`, grouped by `condition_group` so repeated records from one process condition stay together.


In [ ]:
# Initial grouped model evaluation and stability diagnostics
# Input: wobbling_ml_dataset.csv created from the audited Notebook 06 dataset.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Upload the prepared CSV if it is not already in the Colab working directory.
from pathlib import Path
if not Path("wobbling_ml_dataset.csv").exists():
    from google.colab import files
    uploaded = files.upload()
    if "wobbling_ml_dataset.csv" not in uploaded:
        raise FileNotFoundError("Please upload wobbling_ml_dataset.csv")

df = pd.read_csv("wobbling_ml_dataset.csv")

features = [
    "laser_power_W",
    "tangential_wobbling_speed_mm_s",
    "wobbling_diameter_mm"
]
targets = ["bead_width_face_mm", "bead_width_interface_mm"]
required = features + targets + ["condition_group"]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df.dropna(subset=required).copy()
groups = df["condition_group"]

n_groups = groups.nunique()
if n_groups < 5:
    raise ValueError("At least five unique conditions are required for 5-fold GroupKFold.")

print(f"Rows: {len(df)} | Unique conditions: {n_groups}")
print("Condition group sizes:")
display(groups.value_counts().value_counts().sort_index().rename_axis("records_per_condition").to_frame("number_of_conditions"))

cv = GroupKFold(n_splits=5)
models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Ridge regression": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300, min_samples_leaf=2, random_state=42
    )
}

fold_rows = []
oof_rows = []

for target in targets:
    X = df[features]
    y = df[target].astype(float)

    for model_name, model in models.items():
        for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            fitted = model
            fitted.fit(X_train, y_train)
            pred = fitted.predict(X_test)

            fold_rows.append({
                "target": target,
                "model": model_name,
                "fold": fold,
                "train_records": len(train_idx),
                "test_records": len(test_idx),
                "test_conditions": groups.iloc[test_idx].nunique(),
                "MAE_mm": mean_absolute_error(y_test, pred),
                "RMSE_mm": np.sqrt(mean_squared_error(y_test, pred)),
                "R2_fold": r2_score(y_test, pred) if len(test_idx) >= 2 else np.nan
            })

            for row_idx, actual, prediction in zip(test_idx, y_test, pred):
                oof_rows.append({
                    "target": target,
                    "model": model_name,
                    "row_index": int(row_idx),
                    "condition_group": groups.iloc[row_idx],
                    "actual_mm": float(actual),
                    "predicted_mm": float(prediction),
                    "residual_mm": float(actual - prediction)
                })

fold_df = pd.DataFrame(fold_rows)
oof_df = pd.DataFrame(oof_rows)

print("FOLD-BY-FOLD METRICS")
display(fold_df.round(4))

print("POOLED OUT-OF-FOLD METRICS")
pooled_rows = []
for (target, model_name), part in oof_df.groupby(["target", "model"]):
    pooled_rows.append({
        "target": target,
        "model": model_name,
        "MAE_mm": mean_absolute_error(part["actual_mm"], part["predicted_mm"]),
        "RMSE_mm": np.sqrt(mean_squared_error(part["actual_mm"], part["predicted_mm"])),
        "R2_pooled": r2_score(part["actual_mm"], part["predicted_mm"])
    })
pooled_df = pd.DataFrame(pooled_rows)
display(pooled_df.round(4))

print("FOLD METRIC SUMMARY (mean and standard deviation)")
fold_summary = (
    fold_df.groupby(["target", "model"])[["MAE_mm", "RMSE_mm", "R2_fold"]]
    .agg(["mean", "std"])
    .round(4)
)
display(fold_summary)

# Visual diagnostics for the primary exploratory model: Extra Trees / face bead width.
plot_data = oof_df[
    (oof_df["target"] == "bead_width_face_mm") &
    (oof_df["model"] == "Extra Trees")
]
plt.figure(figsize=(6, 6))
plt.scatter(plot_data["actual_mm"], plot_data["predicted_mm"])
lims = [
    min(plot_data["actual_mm"].min(), plot_data["predicted_mm"].min()),
    max(plot_data["actual_mm"].max(), plot_data["predicted_mm"].max())
]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Measured face bead width (mm)")
plt.ylabel("Out-of-fold predicted face bead width (mm)")
plt.title("Grouped CV: Extra Trees — face bead width")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.scatter(plot_data["predicted_mm"], plot_data["residual_mm"])
plt.axhline(0, linestyle="--")
plt.xlabel("Out-of-fold predicted face bead width (mm)")
plt.ylabel("Residual (measured − predicted, mm)")
plt.title("Grouped CV residuals — face bead width")
plt.tight_layout()
plt.show()

fold_df.to_csv("wobbling_fold_metrics.csv", index=False)
pooled_df.to_csv("wobbling_pooled_oof_metrics.csv", index=False)
oof_df.to_csv("wobbling_oof_predictions.csv", index=False)

print("Saved: wobbling_fold_metrics.csv")
print("Saved: wobbling_pooled_oof_metrics.csv")
print("Saved: wobbling_oof_predictions.csv")
